# Chapter 7: Domain Segregation for Agents

Agents derive from the same architectural principles as bounded contexts in DDD.
Without explicit domain boundaries, agents develop **domain envy**: they start
making decisions outside their competence, producing hallucinations and system bugs.

**Concepts:**
- Cognitive Bounded Context (formal 4-tuple definition)
- Scope enforcement at runtime
- Anti-Corruption Layer between domains
- Cross-domain event choreography

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))
from core.context.bounded_context import (
    AgentCapabilityLevel, CognitiveBoundedContext, KnowledgeBase, SemanticContext,
)
print('Domain context module ready.')

## 7.1 Formal Definition of Cognitive Bounded Context

A Cognitive Bounded Context is a 4-tuple **(S, T, K, V)**:

| Symbol | Name | Meaning |
|--------|------|---------|
| **S** | Semantic Context | Domain vocabulary: what concepts the agent understands |
| **T** | Tool Permissions | What operations the agent is permitted to perform |
| **K** | Knowledge Bases | Structured information sources it can access |
| **V** | Value Function | The restricted decision space (optimization target) |

An agent receiving a task outside its **S-T-K-V** must reject or route it.

In [ ]:
import json

# ORDER domain
order_context = CognitiveBoundedContext(
    name='OrderManagement',
    semantic_context=SemanticContext(
        domain='Commerce',
        keywords=['order', 'cart', 'checkout', 'fulfillment', 'shipping', 'line_item'],
        ontology_refs=['schema.org/Order', 'GS1/LogisticLabel'],
    ),
    tool_permissions=[
        AgentCapabilityLevel.READ,
        AgentCapabilityLevel.WRITE,
        AgentCapabilityLevel.EXECUTE,
    ],
    knowledge_bases=[
        KnowledgeBase('orders', 'cosmos://orders', '2.1'),
        KnowledgeBase('product-catalog', 'search://catalog', '1.0'),
    ],
)

# PAYMENT domain (separate context)
payment_context = CognitiveBoundedContext(
    name='PaymentProcessing',
    semantic_context=SemanticContext(
        domain='Finance',
        keywords=['payment', 'invoice', 'refund', 'transaction', 'charge', 'fraud'],
        ontology_refs=['ISO20022', 'PCI-DSS'],
    ),
    tool_permissions=[
        AgentCapabilityLevel.READ,
        AgentCapabilityLevel.EXECUTE,
    ],
    knowledge_bases=[
        KnowledgeBase('transactions', 'postgres://payments', '3.0'),
    ],
)

print('Order context:')
print(json.dumps(order_context.describe(), indent=2))
print()
print('Payment context:')
print(json.dumps(payment_context.describe(), indent=2))

## 7.2 Runtime Scope Enforcement

Call `is_in_scope(task)` **before** any LLM invocation to prevent hallucinations
caused by an agent attempting tasks outside its domain.

In [ ]:
test_tasks = [
    ('Process checkout for cart #4892', order_context, True),
    ('Issue refund for transaction TX-9999', order_context, False),
    ('Validate credit card fraud score', payment_context, True),
    ('Update shipping address for order ORD-001', order_context, True),
    ('Schedule database backup for 2am', order_context, False),
]

print('Scope enforcement results:')
all_correct = True
for task, ctx, expected in test_tasks:
    result = ctx.is_in_scope(task)
    correct = result == expected
    all_correct = all_correct and correct
    icon = '✓' if correct else '✗'
    scope_str = 'IN scope' if result else 'OUT of scope'
    print(f'  {icon} [{ctx.name}] {scope_str}: {task}')

print(f'\nAll enforcement checks correct: {all_correct}')

## 7.3 Capability-Level Permissions

Beyond keyword matching, agents must also have the correct **capability level**:
READ < WRITE < EXECUTE < ORCHESTRATE

In [ ]:
print('Permission checks for Order context:')
for level in AgentCapabilityLevel:
    can = order_context.can_execute(level)
    icon = '✓' if can else '✗'
    print(f'  {icon} {level.name}')

print()
print('Permission checks for Payment context:')
for level in AgentCapabilityLevel:
    can = payment_context.can_execute(level)
    icon = '✓' if can else '✗'
    print(f'  {icon} {level.name}')

# Payment context cannot ORCHESTRATE - by design
print()
print('Payment cannot ORCHESTRATE (correct isolation):', not payment_context.can_execute(AgentCapabilityLevel.ORCHESTRATE))

## 7.4 Cross-Domain Choreography

Domains coordinate via **events** (not direct calls), preserving isolation.
Each domain publishes its own events and subscribes to others' —
the event is the ACL boundary.

```
[Order Domain] ── OrderPlaced ──► [Inventory Domain]
                                     | InventoryReserved
                                     ▼
                 ── InventoryReserved ──► [Payment Domain]
                                     | PaymentProcessed
                                     ▼
                 ── PaymentProcessed ──► [Notification Domain]
```

In [ ]:
import asyncio
from core.messaging.bus import AgentEventBus
from core.messaging.events import Event

trace = []

async def inventory_domain(event: Event):
    trace.append(f'[Inventory] Reacted to {event.event_type}')
    # After processing, emits its own event

async def payment_domain(event: Event):
    trace.append(f'[Payment] Reacted to {event.event_type}')

async def notification_domain(event: Event):
    trace.append(f'[Notification] Reacted to {event.event_type}')

async def demo_choreography():
    bus = AgentEventBus()
    bus.subscribe('OrderPlaced', inventory_domain)
    bus.subscribe('OrderPlaced', payment_domain)
    bus.subscribe('OrderPlaced', notification_domain)

    await bus.publish(Event(event_type='OrderPlaced', payload={'order_id': 'ORD-200'}))
    print('Choreography trace:')
    for step in trace:
        print(f'  {step}')
    print()
    print('Each domain reacted independently - zero coupling between domains.')

asyncio.run(demo_choreography())